In [ ]:
import yfinance as yf
import pandas as pd 
import numpy as np

In [ ]:
# SK하이닉스의 데이터를 수집 
hynix = yf.Ticker("000660.KS")
# 최근 2년간의 데이터를 추출 
df = hynix.history(period='2y')
df.head()

In [ ]:
# 종가, 거래량 데이터만 사용 
df = df[ ['Close', 'Volume'] ]

In [ ]:
# 기술적 지표 : 20일간의 이동평균선 
# 20개의 데이터를 묶어준다. -> rolling(n)
df['MA20'] = df['Close'].rolling(20).mean()
df.iloc[ 15 : 25,  ]

In [ ]:
# 타켓 변수 -> 내일의 종가가 내일의 20일 평균선보다 높은가? (1 : 상승돌파, 0 : 하락)
# 인덱스를 한칸씩 위로 올린다. shift(1) -> 1칸씩 내린다 , shift(-1) -> -1칸씩 이동(위로 1칸 이동)
df['target'] = ( df['Close'].shift(-1) > df['MA20'].shift(-1) ).astype(int)

- astype() -> 타입을 변경하는 함수 (문자형 데이터를 숫자형으로 변경할때는 int)
    - 컬럼 내의 데이터가 '1' -> 1
    - '-' -> error
- to_numeric() -> 범주형 데이터를 수치형을 변경(문자로 되어있는 숫자를 숫자 타입으로 변경 )
    - '1' -> 1
    - '-' -> NaN

In [ ]:
df['target'].value_counts()

In [ ]:
# NLP 감성 점수 데이터를 추가 
# 원래는 기사를 크롤링하여 감성 점수를 예측하고 입력해야하지만 
# 무작위한 데이터를 입력 
df['NLP_Sentiment'] = np.random.uniform(-1, 1, len(df))

In [ ]:
df['NLP_Sentiment'].describe()

In [ ]:
# !pip install OpenDartReader

In [ ]:
# 영업이익 데이터를 추가 
from opendartreader import OpenDartReader
from datetime import datetime
import os
from dotenv import load_dotenv

In [ ]:
# .env 파일에서 환경 변수 로드 
load_dotenv()

In [ ]:
api_key = os.getenv('api_key')

In [ ]:
df.head(1)

In [ ]:
# 시차 정보를 제거 -> 시차 정보가 없는 시계열 데이터와 시차 정보가 존재하는 시계열 데이터가 결합 x
df.index = df.index.tz_localize(None)
df.head(1)


In [ ]:
# dart에서 데이터를 수집 
dart = OpenDartReader(api_key)

In [25]:
# 현재년도 로드 -> 최근 3년간 목록을 생성 
current_year = datetime.now().year
years = [ current_year - 2, current_year - 1, current_year ]
years

[2024, 2025, 2026]

In [ ]:
# 분기 보고서 codes
reprt_codes = ['11013', '11012', '11014', '11011']
dart_data_list = []

In [27]:
for year in years:
    for code in reprt_codes:
        try : 
            report = dart.finstate('000660', year, code)
            if report is not None:
                op_profit = report[ (report['fs_div'] == 'CFS')  & ( report['account_nm'] == '영업이익' ) ]
                if not op_profit.empty:
                    val = int(
                            op_profit['thstrm_amount'].values[0].replace(',', '')
                        )
                    # code 11013 -> 1분기 보고서 
                    if code == '11013' : d = f"{year}-05-15"
                    elif code == '11012' : d = f"{year}-08-14"
                    elif code == '11014' : d = f"{year}-11-14"
                    else : d = f"{year+1}-03-31"

                    report_date = pd.to_datetime(d)
                    if report_date <= datetime.now():
                        dart_data_list.append({'Date' : report_date, 'Operation_Profit' : val})
        except: continue
dart_data_list

{'status': '013', 'message': '조회된 데이타가 없습니다.'}

{'status': '013', 'message': '조회된 데이타가 없습니다.'}

{'status': '013', 'message': '조회된 데이타가 없습니다.'}



[{'Date': Timestamp('2024-05-15 00:00:00'), 'Operation_Profit': 2886029000000},
 {'Date': Timestamp('2024-08-14 00:00:00'), 'Operation_Profit': 5468536000000},
 {'Date': Timestamp('2024-11-14 00:00:00'), 'Operation_Profit': 7029958000000},
 {'Date': Timestamp('2025-03-31 00:00:00'),
  'Operation_Profit': 23467319000000},
 {'Date': Timestamp('2025-05-15 00:00:00'), 'Operation_Profit': 7440504000000},
 {'Date': Timestamp('2025-08-14 00:00:00'), 'Operation_Profit': 9212851000000},
 {'Date': Timestamp('2025-11-14 00:00:00'),
  'Operation_Profit': 11383390000000},
 {'Date': Timestamp('2026-03-31 00:00:00'),
  'Operation_Profit': 47206319000000},
 {'Date': Timestamp('2026-05-15 00:00:00'),
  'Operation_Profit': 37610283000000}]